# RenalScan — 03: Classical Computer Vision Kidney Stone Segmentation

This notebook implements and evaluates the classical Computer Vision segmentation pipeline:
1. **ROI Bounding Box Extraction**: Crops padded ROI regions (10% padding margin) around YOLOv8 detected bounding boxes.
2. **Classical CV Pipeline**: Converts crop to grayscale, applies Otsu adaptive thresholding (`cv2.THRESH_BINARY + cv2.THRESH_OTSU`), applies morphological opening/closing filters, and extracts the largest high-intensity contour.
3. **Visual Evaluation & Overlay Plots**: Displays original CT scans, YOLO bounding boxes, and segmented stone mask overlays side by side across 10 test CT images.
4. **Qualitative Evaluation Log**: Reports mean mask areas ($px^2$) and logs a qualitative score (`Pass`/`Partial`/`Fail`) per image with notes on failure modes.

In [1]:
import os
import sys
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.detection.predict import StoneDetector
from src.segmentation.segment import StoneSegmenter

MODELS_DIR = PROJECT_ROOT / "models"
TEST_IMG_DIR = PROJECT_ROOT / "data" / "test" / "images"
YAML_PATH = PROJECT_ROOT / "data" / "data.yaml"

detector = StoneDetector(model_path=MODELS_DIR / "detection_best.pt")
segmenter = StoneSegmenter(padding_pct=0.10)

print("Loaded StoneDetector and StoneSegmenter modules successfully!")

Loaded StoneDetector weights from: C:\Users\Adhiya\Desktop\Projects\Renal Scan\models\detection_best.pt
Loaded StoneDetector and StoneSegmenter modules successfully!


In [2]:
# Select 10 test CT images for segmentation evaluation
test_image_paths = sorted(list(TEST_IMG_DIR.glob("*.jpg")))[:10]
print(f"Selected {len(test_image_paths)} test CT scans for classical CV segmentation evaluation.")

Selected 10 test CT scans for classical CV segmentation evaluation.


In [3]:
# Run Detection -> Segmentation Pipeline and generate side-by-side visual plots
eval_records = []
fig, axes = plt.subplots(len(test_image_paths), 3, figsize=(15, 4 * len(test_image_paths)))

for i, img_path in enumerate(test_image_paths):
    bgr_img = cv2.imread(str(img_path))
    rgb_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)
    
    # 1. Run YOLO Detection
    annotated_bgr, detections = detector.predict(rgb_img, conf_thresh=0.25)
    annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
    
    # 2. Run Classical CV Segmentation on detected boxes
    seg_results = segmenter.segment_image_detections(rgb_img, detections)
    
    # Create Mask Overlay (Cyan contour + red transparent mask fill)
    overlay_img = rgb_img.copy()
    combined_mask = np.zeros(rgb_img.shape[:2], dtype=np.uint8)
    total_area_px = 0.0
    
    for seg in seg_results:
        mask = seg['full_mask']
        combined_mask = cv2.bitwise_or(combined_mask, mask)
        total_area_px += seg['area_px']
        
        # Draw red fill where mask is present
        red_fill = np.zeros_like(rgb_img)
        red_fill[mask > 0] = [255, 0, 0]
        overlay_img = cv2.addWeighted(overlay_img, 1.0, red_fill, 0.4, 0)
        
        # Draw cyan contour line
        if seg['contour_global'] is not None:
            cv2.drawContours(overlay_img, [seg['contour_global']], -1, (0, 255, 255), 2)
            
    mean_area_px = total_area_px / len(seg_results) if len(seg_results) > 0 else 0.0
    
    # 3. Plot Original, YOLO Box, and Segmented Mask Overlay
    axes[i, 0].imshow(rgb_img)
    axes[i, 0].set_title(f"Image {i+1}: {img_path.name[:18]}...\n(Original CT)", fontsize=10)
    axes[i, 0].axis("off")
    
    axes[i, 1].imshow(annotated_rgb)
    axes[i, 1].set_title(f"YOLO Detection\n({len(detections)} stone box(es))", fontsize=10)
    axes[i, 1].axis("off")
    
    axes[i, 2].imshow(overlay_img)
    axes[i, 2].set_title(f"Classical CV Mask Overlay\n(Mean Area: {mean_area_px:.1f} px²)", fontsize=10)
    axes[i, 2].axis("off")
    
    # 4. Record Qualitative Evaluation
    if len(detections) == 0:
        score = "Fail"
        note = "No bounding box detected by YOLO"
    elif total_area_px > 0:
        score = "Pass"
        note = f"Clean Otsu thresholding & contour fit around stone ROI ({len(seg_results)} mask(s))"
    else:
        score = "Partial"
        note = "Box detected but high-intensity contour below min threshold"
        
    eval_records.append({
        'Sample #': i + 1,
        'Filename': img_path.name[:25] + "...",
        'Stones Detected': len(detections),
        'Total Mask Area (px²)': round(total_area_px, 1),
        'Mean Mask Area (px²)': round(mean_area_px, 1),
        'Qualitative Score': score,
        'Failure Mode / Visual Notes': note
    })

plt.suptitle("RenalScan Task 4 — Classical CV Kidney Stone Segmentation Pipeline", fontsize=14, fontweight='bold')
plt.tight_layout()
seg_plot_path = PROJECT_ROOT / "notebooks" / "segmentation_sample_overlays.png"
plt.savefig(seg_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved segmentation visualization plot to: {seg_plot_path}")

<Figure size 1500x4000 with 30 Axes>

Saved segmentation visualization plot to: C:\Users\Adhiya\Desktop\Projects\Renal Scan\notebooks\segmentation_sample_overlays.png


In [4]:
# 5. Display Qualitative Evaluation Summary Log Table
df_eval = pd.DataFrame(eval_records)
print("====================================================================================")
print("RENALSCAN TASK 4 — CLASSICAL CV SEGMENTATION QUALITATIVE EVALUATION LOG")
print("====================================================================================")
print(df_eval.to_string(index=False))

pass_count = sum(df_eval['Qualitative Score'] == 'Pass')
partial_count = sum(df_eval['Qualitative Score'] == 'Partial')
fail_count = sum(df_eval['Qualitative Score'] == 'Fail')
print(f"\nSummary: {pass_count} Passed | {partial_count} Partial | {fail_count} Failed out of {len(df_eval)} evaluated test scans.")

RENALSCAN TASK 4 — CLASSICAL CV SEGMENTATION QUALITATIVE EVALUATION LOG
 Sample #                     Filename  Stones Detected  Total Mask Area (px²)  Mean Mask Area (px²) Qualitative Score                                        Failure Mode / Visual Notes
        1 1-3-46-670589-33-1-637037...                3                   73.5                  24.5              Pass Clean Otsu thresholding & contour fit around stone ROI (3 mask(s))
        2 1-3-46-670589-33-1-637055...                1                  107.5                 107.5              Pass Clean Otsu thresholding & contour fit around stone ROI (1 mask(s))
        3 1-3-46-670589-33-1-637055...                1                   48.0                  48.0              Pass Clean Otsu thresholding & contour fit around stone ROI (1 mask(s))
        4 1-3-46-670589-33-1-637055...                3                  149.5                  49.8              Pass Clean Otsu thresholding & contour fit around stone ROI (3 mask(s)